# Day3_09 · 평균의 불확실성 확인

표본을 다시 뽑을 때 평균이 얼마나 달라지는지 그림과 신뢰구간으로 확인합니다.

## 이 노트북에서 확인할 내용

- 표본에 따라 평균이 달라지는 모습을 확인할 수 있습니다.
- bootstrap 평균 분포와 95% 신뢰구간을 만들 수 있습니다.
- 표본 수와 구간 폭의 관계를 설명할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 한 개씩 실행하고 바로 다음 결과를 확인합니다. 수정 안내가 있는 곳에서는 표시된 값 하나만 바꾼 뒤 결과를 비교합니다.

### D3B-09-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-09-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-09-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-09-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-09-05 · pandas와 numpy 불러오기

In [ ]:
import pandas as pd
import numpy as np

### D3B-09-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### D3B-09-07 · 콘크리트 강도자료 읽기

In [ ]:
concrete = pd.read_csv(DATA_DIR / 'concrete_strength.csv')

In [ ]:
concrete.head()

### D3B-09-08 · 압축강도 열 선택

In [ ]:
strength = concrete['strength_mpa'].dropna()

In [ ]:
strength.head()

### D3B-09-09 · 전체 자료의 평균 확인

In [ ]:
strength.mean()

### D3B-09-10 · 전체 자료의 분포 확인

In [ ]:
strength.describe().round(2)

## 일부 행만 뽑으면 평균이 달라지는가?

전체 자료에서 같은 수의 행을 다시 뽑더라도 어떤 행이 선택되느냐에 따라 평균은 조금씩 달라집니다.

### D3B-09-11 · 10개 행을 한 번 뽑기

In [ ]:
sample_10_a = strength.sample(n=10, random_state=1)

In [ ]:
sample_10_a

### D3B-09-12 · 첫 표본의 평균 확인

In [ ]:
sample_10_a.mean()

### D3B-09-13 · 다른 10개 행 뽑기

In [ ]:
sample_10_b = strength.sample(n=10, random_state=2)

In [ ]:
sample_10_b

### D3B-09-14 · 두 번째 표본의 평균 확인

In [ ]:
sample_10_b.mean()

### D3B-09-15 · 표본 평균 두 개 비교

In [ ]:
two_sample_means = pd.Series({'seed 1': sample_10_a.mean(), 'seed 2': sample_10_b.mean(), 'all rows': strength.mean()})

In [ ]:
two_sample_means.round(2)

## 같은 크기의 표본을 여러 번 만들기

### D3B-09-16 · 무작위 수 생성기 준비

In [ ]:
rng = np.random.default_rng(42)

In [ ]:
rng

### D3B-09-17 · 크기 20인 표본 평균 1,000개 만들기

In [ ]:
means_n20 = pd.Series([rng.choice(strength, size=20, replace=True).mean() for _ in range(1000)], name='mean_strength')

In [ ]:
means_n20.head()

### D3B-09-18 · 1,000개 평균의 분포 그리기

In [ ]:
sns.histplot(means_n20, bins=30, kde=True)
plt.axvline(strength.mean(), color='red', linestyle='--', label='All-row mean')
plt.xlabel('Resampled mean strength (MPa)')
plt.legend()
plt.show()

### D3B-09-19 · 가운데 95% 범위 계산

In [ ]:
interval_n20 = means_n20.quantile([0.025, 0.975])

In [ ]:
interval_n20.round(2)

무작위로 다시 뽑은 평균 1,000개의 가운데 95% 구간을 확인했습니다. 전문가들은 이처럼 관측값을 복원추출해 통계량의 불확실성을 살피는 방법을 bootstrap이라고 부릅니다.

### D3B-09-20 · 크기 100인 표본 평균 1,000개 만들기

In [ ]:
rng = np.random.default_rng(42)
means_n100 = pd.Series([rng.choice(strength, size=100, replace=True).mean() for _ in range(1000)], name='mean_strength')

In [ ]:
means_n100.head()

### D3B-09-21 · 크기 100의 95% 범위 계산

In [ ]:
interval_n100 = means_n100.quantile([0.025, 0.975])

In [ ]:
interval_n100.round(2)

### D3B-09-22 · 두 표본 크기의 구간 폭 비교

In [ ]:
interval_widths = pd.Series({
    'sample size 20': interval_n20.loc[0.975] - interval_n20.loc[0.025],
    'sample size 100': interval_n100.loc[0.975] - interval_n100.loc[0.025],
})

In [ ]:
interval_widths.round(2)

### D3B-09-23 · 두 평균 분포 겹쳐 보기

In [ ]:
sns.kdeplot(means_n20, fill=True, alpha=0.25, label='n=20')
sns.kdeplot(means_n100, fill=True, alpha=0.25, label='n=100')
plt.xlabel('Resampled mean strength (MPa)')
plt.legend()
plt.show()

표본 수가 커지면 평균의 흔들림이 보통 줄어듭니다. 구간이 좁다는 사실은 자료가 편향되지 않았거나 인과관계가 확인되었다는 뜻은 아닙니다.

### D3B-09-24 · SciPy의 bootstrap 함수 불러오기

In [ ]:
from scipy.stats import bootstrap

### D3B-09-25 · 평균의 bootstrap 신뢰구간 계산

In [ ]:
bootstrap_result = bootstrap((strength.to_numpy(),), np.mean, n_resamples=3000, confidence_level=0.95, rng=np.random.default_rng(42))

In [ ]:
bootstrap_result.confidence_interval

### D3B-09-26 · 결과 표 만들기

In [ ]:
bootstrap_summary = pd.Series({
    'observed_mean_mpa': strength.mean(),
    'lower_95_mpa': bootstrap_result.confidence_interval.low,
    'upper_95_mpa': bootstrap_result.confidence_interval.high,
})

In [ ]:
bootstrap_summary.round(2)

### D3B-09-27 · 직접 바꿀 표본 크기 정하기

In [ ]:
# 30을 50 또는 200으로 바꾸어 보세요.
chosen_sample_size = 30

In [ ]:
chosen_sample_size

### D3B-09-28 · 선택한 크기의 평균 분포 만들기

In [ ]:
rng = np.random.default_rng(7)
chosen_means = pd.Series([rng.choice(strength, size=chosen_sample_size, replace=True).mean() for _ in range(1000)])

In [ ]:
chosen_means.describe().round(2)

### D3B-09-29 · 선택한 평균 분포 그리기

In [ ]:
sns.histplot(chosen_means, bins=30, kde=True)
plt.xlabel('Resampled mean strength (MPa)')
plt.title(f'Sample size = {chosen_sample_size}')
plt.show()

### D3B-09-CODEX · Codex에 확장 분석 요청하기

In [ ]:
codex_request = '''기존 셀은 수정하지 마세요. strength Series를 사용해 중앙값의 bootstrap 95% 신뢰구간을 계산하는 짧은 코드 셀을 제안해 주세요. 평균 결과와 중앙값 결과를 한 표로 비교하고, 두 구간이 답하는 질문이 어떻게 다른지 설명해 주세요. 난수는 고정하세요.'''.strip()

In [ ]:
codex_request

### D3B-09-CHECK · 제안받은 코드를 실행하기 전에 확인하기

In [ ]:
codex_checklist = pd.Series(['strength를 그대로 사용하는가?', '난수가 고정되어 있는가?', '평균과 중앙값을 혼동하지 않는가?'], name='확인할 내용')

In [ ]:
codex_checklist